In [0]:
# ============================================================
# CAPA SILVER: Limpieza y deduplicación - VERSIÓN PRO
# ============================================================
from pyspark.sql.functions import col, current_timestamp, row_number
from pyspark.sql.window import Window

print("📥 Leyendo tabla bronze_telemetry...")
df_bronze = spark.read.table("bronze_telemetry")
rows_before = df_bronze.count()
print(f"✅ Filas en Bronze: {rows_before}")

# 1. DEDUPLICAR PRIMERO - nos quedamos con el último ingest por event_id
print("\n🔁 Deduplicando por event_id (último en llegar gana)...")
window_spec = Window.partitionBy("event_id").orderBy(col("ingest_timestamp").desc())
df_dedup = df_bronze.withColumn("rn", row_number().over(window_spec)) \
                    .filter(col("rn") == 1).drop("rn")

rows_after_dedup = df_dedup.count()
print(f"Duplicados eliminados: {rows_before - rows_after_dedup}")

# 2. FILTROS DE CALIDAD (tu misma lógica, intacta)
print("\n🧹 Aplicando reglas de calidad...")
df_silver = df_dedup.filter(
    (col("event_id").isNotNull()) &
    (col("device_id").isNotNull()) &
    (col("speed_kmh") >= 0) & (col("speed_kmh") <= 200) &
    (col("engine_temp_c") >= -50) & (col("engine_temp_c") <= 150) &
    (col("battery_pct") >= 0) & (col("battery_pct") <= 100)
)

rows_final = df_silver.count()
print(f"✅ Filas después de filtros: {rows_final}")
print(f"🗑  Descartadas por calidad: {rows_after_dedup - rows_final}")

# 3. Columna de auditoría Silver
df_silver_final = df_silver.withColumn("silver_processed_at", current_timestamp())

print("\n📊 Vista previa Silver:")
display(df_silver_final.limit(5))

# 4. Guardar
print("\n💾 Guardando silver_telemetry...")
df_silver_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_telemetry")

print(f"\n✅ ¡ÉXITO! {rows_before} → {rows_final} filas ({round(rows_final/rows_before*100,2)}% retenidas)")


In [0]:
# Celda 2 para que quede igual que en Bronze
# display(spark.sql("DESCRIBE HISTORY silver_telemetry"))